In [1]:
import pandas as pd
import numpy as np
import polars as pl

In [2]:
df = pl.read_csv("./data/aapl_24hr.csv")
# Ensure TIME_M is parsed as a datetime column with strict=False to handle invalid formats
df = df.with_columns(
    pl.col("TIME_M").str.to_time(format="%H:%M:%S%.f").alias("TIME_M")
)

# Filter rows between 9:30:00 and 16:00:00
df = df.filter(
    pl.col("TIME_M").is_between(pl.time(9, 30), pl.time(16, 0))
)

df.shape

(11517550, 14)

In [3]:
df['QU_CANCEL'].value_counts()

QU_CANCEL,count
str,u32
null,11517550


# TAQ Quote - Data Dictionary
Exchange that issued the quote (EX)

Bid price (BID)

Bid size in units of trade (BIDSIZ)

Ask price (ASK)

Ask size in units of trade (ASKSIZ)

Condition of quote issued (QU_COND)

Bid exchange (BIDEX)

Ask exchange (ASKEX)

Quote Sequence Number (QU_SEQNUM)

National BBO Indicator (NATBBO_IND)

NASD BBO Indicator (NASDBBO_IND)

Quote Cancel/Correction (QU_CANCEL)

Source of Quote (QU_SOURCE)

## Quote Condition
‘R’ = Regular, two-sided open quote

‘L’ = Closed Market Maker (NASD)

‘Y’ = Regular - One Sided Quote (NASDAQ)

# single exchange
1. find the most liquid exchange ==> narrow apread
    - filter all two-sided quotes and get the narrowest average for each exchange
2. filter for liquid exchange

In [4]:
df['QU_COND'].value_counts()

QU_COND,count
str,u32
"""R""",11517548
"""Y""",2


In [5]:
# Filter for rows where QU_COND is 'R'
filtered_df = df.filter(pl.col("QU_COND") == "R")

# Create a new column 'spread' calculated as ASK - BID
filtered_df = filtered_df.with_columns((pl.col("ASK") - pl.col("BID")).alias("spread"))

# Aggregate by EX column and calculate the average spread
average_spread = (
	filtered_df
	.group_by("EX")
	.agg(pl.col("spread").mean().alias("average_spread"),
        pl.col("spread").count().alias("count"))
    .sort("count")
)

average_spread

EX,average_spread,count
str,f64,u32
"""C""",1.455613,3287
"""A""",0.952236,98952
"""B""",0.416292,210924
"""M""",0.585905,283239
"""H""",0.114937,351315
…,…,…
"""N""",0.024989,1104854
"""U""",0.019057,1326468
"""Z""",0.015741,1374482


*** EX Q (NASDAQ) has the tightest spread, also has the highest refresh. We assume we can connect directly with NASDAQ ***

In [6]:
filtered_df.filter(pl.col("EX").is_in(["Q"]))['spread'].describe()
# filtered_df.filter(pl.col("EX").is_in(["K"]))['spread'].describe()
# filtered_df.filter(pl.col("EX").is_in(["Z"]))['spread'].describe()

statistic,value
str,f64
"""count""",1.596396e6
"""null_count""",0.0
"""mean""",0.014484
"""std""",0.005745
"""min""",0.01
"""25%""",0.01
"""50%""",0.01
"""75%""",0.02
"""max""",0.14


In [7]:
del filtered_df
del average_spread

# Data prep
1. filter trading hours
2. Nasdaq quotes
3. take twosided quotes only

In [10]:
# filter trading hours
# df = df.with_columns(
#     pl.col("TIME_M").str.to_time(format="%H:%M:%S%.f").alias("TIME_M")
# )
df = df.filter(
    pl.col("TIME_M").is_between(pl.time(9, 30), pl.time(16, 0))
)
# Filter for Nasdaq exchange
df = df.filter(pl.col("EX") == "Q")
# Filter for rows where QU_COND is 'R'
df = df.filter(pl.col("QU_COND") == "R")
df.shape

(1596396, 14)

In [11]:
df.describe()

statistic,DATE,TIME_M,EX,BID,BIDSIZ,ASK,ASKSIZ,QU_COND,QU_SEQNUM,NATBBO_IND,QU_CANCEL,QU_SOURCE,SYM_ROOT,SYM_SUFFIX
str,str,str,str,f64,f64,f64,f64,str,f64,f64,str,str,str,str
"""count""","""1596396""","""1596396""","""1596396""",1.596396e6,1.596396e6,1.596396e6,1.596396e6,"""1596396""",1.596396e6,1.596396e6,"""0""","""1596396""","""1596396""","""0"""
"""null_count""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,"""0""",0.0,0.0,"""1596396""","""0""","""0""","""1596396"""
"""mean""",null,"""12:27:54.730227""",null,173.016099,5.421533,173.030584,5.599605,null,4.2449e7,1.65862,null,null,null,null
"""std""",null,null,null,0.450201,4.291643,0.450235,8.026006,null,2.2900e7,1.505034,null,null,null,null
"""min""","""2023-05-10""","""09:30:00.000431""","""Q""",171.9,1.0,171.91,1.0,"""R""",1.095038e6,0.0,null,"""N""","""AAPL""",null
"""25%""",null,"""10:45:28.094228""",null,172.72,3.0,172.74,3.0,null,2.3702965e7,0.0,null,null,null,null
"""50%""",null,"""12:12:57.029143""",null,173.02,5.0,173.04,5.0,null,4.1735538e7,2.0,null,null,null,null
"""75%""",null,"""14:07:26.259826""",null,173.36,7.0,173.38,7.0,null,6.1941918e7,2.0,null,null,null,null
"""max""","""2023-05-10""","""15:59:59.997234""","""Q""",174.03,212.0,174.04,637.0,"""R""",8.3998381e7,4.0,null,"""N""","""AAPL""",null


In [12]:
df['NATBBO_IND'].value_counts()

NATBBO_IND,count
i64,u32
0,611505
2,645875
4,339016


### National Best Bid-Offer (NBBO) - Legend 
‘0’ = No National BBO change - Current quote does not affect the BBO. No National appendage is required.

‘1’ = No National BBO Can be Calculated- The National BBO cannot be calculated therefore vendors should show National BBO fields as blank. No Appendage is required.

‘2’ = Short Form National BBO Appendage Attached – A new National BBO was generated as a result of the UTP participant’s quote update and the new information is contained in the short form appendage (NBBO FILE)

‘3’ = Long Format of National BBO Appendage - A new National BBO is generated and the new BBO information is contained in the Long National BBO appendage (NBBO FILE)

‘4’ = Quote Contains all NASD BBO Information - Current quote is itself the new NASD BBO. No NASD appendage is required.

---
In the case above, I will track all bid offer and size regardless of NATBBO indicator

In [13]:
df['QU_COND'].value_counts()

QU_COND,count
str,u32
"""R""",1596396


In [15]:
df.columns

['DATE',
 'TIME_M',
 'EX',
 'BID',
 'BIDSIZ',
 'ASK',
 'ASKSIZ',
 'QU_COND',
 'QU_SEQNUM',
 'NATBBO_IND',
 'QU_CANCEL',
 'QU_SOURCE',
 'SYM_ROOT',
 'SYM_SUFFIX']

# Try OBI strategy

In [23]:
class OBIVWAPStrategy:
    def __init__(self, vwap_window: int, obi_threshold: float, initial_cash: float = 100_000):
        self.vwap_window = vwap_window
        self.obi_threshold = obi_threshold
        self.cash = initial_cash
        self.position = 0
        self.account_balance = []

    def calculate_vwap(self, df: pl.DataFrame) -> pl.DataFrame:
        # Calculate MID_PRICE
        df = df.with_columns(
            ((pl.col("BID") + pl.col("ASK")) / 2).alias("MID_PRICE")
        )
        # Calculate Volume
        df = df.with_columns(
            (pl.col("BIDSIZ") + pl.col("ASKSIZ")).alias("Volume")
        )
        # Calculate VWAP using rolling window
        df = df.with_columns(
            (
                (pl.col("MID_PRICE") * pl.col("Volume"))
                .rolling_sum(window_size=self.vwap_window)
                / pl.col("Volume").rolling_sum(window_size=self.vwap_window)
            ).alias("VWAP")
        )
        return df

    def calculate_obi(self, df: pl.DataFrame) -> pl.DataFrame:
        # Calculate Order Book Imbalance (OBI)
        df = df.with_columns(
            (
                (pl.col("BIDSIZ") - pl.col("ASKSIZ"))
                / (pl.col("BIDSIZ") + pl.col("ASKSIZ"))
            ).alias("OBI")
        )
        return df

    def generate_signals(self, df: pl.DataFrame) -> pl.DataFrame:
        # Calculate VWAP and OBI
        df = self.calculate_vwap(df)
        df = self.calculate_obi(df)

        # Generate signals based on OBI threshold
        df = df.with_columns(
            pl.when(pl.col("OBI") > self.obi_threshold)
            .then(1)  # Buy signal
            .when(pl.col("OBI") < -self.obi_threshold)
            .then(-1)  # Sell signal
            .otherwise(0)  # No signal
            .alias("Signal")
        )
        return df

    def backtest(self, df: pl.DataFrame) -> pl.DataFrame:
        # Initialize account balance tracking
        account_balance = []

        # Iterate over rows to simulate trading
        for row in df.iter_rows(named=True):
            if row["Signal"] == 1 and self.cash >= row["ASK"] * 100 and self.position <= 1:
                # Buy 100 shares
                self.position = 100
                self.cash -= row["ASK"] * 100
            elif row["Signal"] == -1 and self.position > -1:
                # Sell 100 shares
                self.position = -100
                self.cash += row["BID"] * 100
            elif row["Signal"] == 0 and self.position != 0:
                # Close position
                if self.position > 0:
                    self.cash += row["BID"] * self.position
                else:
                    self.cash -= row["ASK"] * abs(self.position)
                self.position = 0

            # Record account balance
            account_balance.append(self.cash + self.position * (row["ASK"] if self.position > 0 else row["BID"]))

        # Add account balance to the DataFrame
        df = df.with_columns(pl.Series("Account_Balance", account_balance))
        return df

In [24]:
strategy = OBIVWAPStrategy(vwap_window=50, obi_threshold=0.05)
signal_data = strategy.generate_signals(df)
backtest_data = strategy.backtest(signal_data)

print(backtest_data.head())


shape: (5, 20)
┌────────────┬────────────────────┬─────┬────────┬───┬──────┬───────────┬────────┬─────────────────┐
│ DATE       ┆ TIME_M             ┆ EX  ┆ BID    ┆ … ┆ VWAP ┆ OBI       ┆ Signal ┆ Account_Balance │
│ ---        ┆ ---                ┆ --- ┆ ---    ┆   ┆ ---  ┆ ---       ┆ ---    ┆ ---             │
│ str        ┆ time               ┆ str ┆ f64    ┆   ┆ f64  ┆ f64       ┆ i32    ┆ f64             │
╞════════════╪════════════════════╪═════╪════════╪═══╪══════╪═══════════╪════════╪═════════════════╡
│ 2023-05-10 ┆ 09:30:00.000431616 ┆ Q   ┆ 172.98 ┆ … ┆ null ┆ -0.888889 ┆ -1     ┆ 100000.0        │
│ 2023-05-10 ┆ 09:30:00.000916688 ┆ Q   ┆ 172.98 ┆ … ┆ null ┆ -0.837838 ┆ -1     ┆ 100000.0        │
│ 2023-05-10 ┆ 09:30:00.000949802 ┆ Q   ┆ 172.98 ┆ … ┆ null ┆ -0.7      ┆ -1     ┆ 100000.0        │
│ 2023-05-10 ┆ 09:30:00.001405909 ┆ Q   ┆ 172.98 ┆ … ┆ null ┆ -0.666667 ┆ -1     ┆ 100000.0        │
│ 2023-05-10 ┆ 09:30:00.002053465 ┆ Q   ┆ 172.98 ┆ … ┆ null ┆ -0.578947 ┆ -1

In [26]:
backtest_data.tail(10)

DATE,TIME_M,EX,BID,BIDSIZ,ASK,ASKSIZ,QU_COND,QU_SEQNUM,NATBBO_IND,QU_CANCEL,QU_SOURCE,SYM_ROOT,SYM_SUFFIX,MID_PRICE,Volume,VWAP,OBI,Signal,Account_Balance
str,time,str,f64,i64,f64,i64,str,i64,i64,str,str,str,str,f64,i64,f64,f64,i32,f64
"""2023-05-10""",15:59:59.901133127,"""Q""",173.54,31,173.56,56,"""R""",83996333,4,null,"""N""","""AAPL""",null,173.55,87,173.547911,-0.287356,-1,1.2328442e7
"""2023-05-10""",15:59:59.901347442,"""Q""",173.55,1,173.56,56,"""R""",83996368,0,null,"""N""","""AAPL""",null,173.555,57,173.548065,-0.964912,-1,1.2328441e7
"""2023-05-10""",15:59:59.903542325,"""Q""",173.55,1,173.56,55,"""R""",83996618,4,null,"""N""","""AAPL""",null,173.555,56,173.548204,-0.964286,-1,1.2328441e7
"""2023-05-10""",15:59:59.903542715,"""Q""",173.55,1,173.56,30,"""R""",83996619,4,null,"""N""","""AAPL""",null,173.555,31,173.548277,-0.935484,-1,1.2328441e7
"""2023-05-10""",15:59:59.903648595,"""Q""",173.55,1,173.56,31,"""R""",83996631,2,null,"""N""","""AAPL""",null,173.555,32,173.548354,-0.9375,-1,1.2328441e7
"""2023-05-10""",15:59:59.922348038,"""Q""",173.54,31,173.56,31,"""R""",83997195,2,null,"""N""","""AAPL""",null,173.55,62,173.54836,0.0,0,1.232844e7
"""2023-05-10""",15:59:59.922392124,"""Q""",173.55,1,173.56,31,"""R""",83997196,0,null,"""N""","""AAPL""",null,173.555,32,173.548409,-0.9375,-1,1.232844e7
"""2023-05-10""",15:59:59.944550801,"""Q""",173.55,58,173.56,31,"""R""",83997486,4,null,"""N""","""AAPL""",null,173.555,89,173.548518,0.303371,1,1.2345795e7
"""2023-05-10""",15:59:59.980121311,"""Q""",173.55,58,173.56,42,"""R""",83998027,4,null,"""N""","""AAPL""",null,173.555,100,173.548728,0.16,1,1.2345795e7


In [27]:
backtest_data["Account_Balance"].describe()

statistic,value
str,f64
"""count""",1.596396e6
"""null_count""",0.0
"""mean""",8.4128e6
"""std""",3.0961e6
"""min""",8416.0
"""25%""",5.905426e6
"""50%""",9.407712e6
"""75%""",1.1141611e7
"""max""",1.2937458e7


In [20]:
backtest_data['OBI'].sum()

9800.290018223943

In [21]:
backtest_data.tail(5)

DATE,TIME_M,EX,BID,BIDSIZ,ASK,ASKSIZ,QU_COND,QU_SEQNUM,NATBBO_IND,QU_CANCEL,QU_SOURCE,SYM_ROOT,SYM_SUFFIX,MID_PRICE,Volume,VWAP,OBI,Signal,Account_Balance
str,time,str,f64,i64,f64,i64,str,i64,i64,str,str,str,str,f64,i64,f64,f64,i32,f64
"""2023-05-10""",15:59:59.922348038,"""Q""",173.54,31,173.56,31,"""R""",83997195,2,null,"""N""","""AAPL""",null,173.55,62,173.54836,0.0,0,1.232844e7
"""2023-05-10""",15:59:59.922392124,"""Q""",173.55,1,173.56,31,"""R""",83997196,0,null,"""N""","""AAPL""",null,173.555,32,173.548409,-0.9375,-1,1.2328e7
"""2023-05-10""",15:59:59.944550801,"""Q""",173.55,58,173.56,31,"""R""",83997486,4,null,"""N""","""AAPL""",null,173.555,89,173.548518,0.303371,1,1.2346e7
"""2023-05-10""",15:59:59.980121311,"""Q""",173.55,58,173.56,42,"""R""",83998027,4,null,"""N""","""AAPL""",null,173.555,100,173.548728,0.16,1,1.2346e7
"""2023-05-10""",15:59:59.997234402,"""Q""",173.55,58,173.56,76,"""R""",83998381,4,null,"""N""","""AAPL""",null,173.555,134,173.549005,-0.134328,-1,1.2328e7


In [22]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Convert to pandas
pdf = backtest_data.to_pandas()
pdf["TIME_M"] = pd.to_datetime(pdf["DATE"] + " " + pdf["TIME_M"].astype(str), format='mixed')

# Plot
for ticker in set(pdf['SYM_ROOT']):
    ticker_df = pdf[pdf['SYM_ROOT'] == ticker]
    plt.figure(figsize=(12, 6))
    plt.plot(ticker_df["TIME_M"], ticker_df["Account_Balance"])
    plt.xlabel("TIME_M")
    plt.ylabel("Account_Balance")
    plt.title(f"Account Balance for {ticker}")
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    plt.xticks(rotation=45)
    plt.show()

ArrowInvalid: Value 34200000431616 has non-zero nanoseconds